In [2]:
import os
import shutil
from pathlib import Path

# Combine parallel inferred cNMF component results into one run

In [ ]:

def rename_and_move_files_NMF(file_name_input, file_name_output, source_folder, destination_folder, len = 10):

    # Create destination folder if it doesn't exist
    Path(destination_folder).mkdir(parents=True, exist_ok=True)
    
    processed_files = []
    
    # Process NMF files for i 
    for i in range(len):
        source_filename = f"{file_name_input}_{i}.df.npz"
        source_path = os.path.join(source_folder, source_filename)
        
        # Check if source file exists
        if os.path.exists(source_path):
            
            # Create new filename
            new_filename = f"{file_name_output}_{i}.df.npz"
            destination_path = os.path.join(destination_folder, new_filename)
            
            try:
                # Copy and rename file to destination
                shutil.copy2(source_path, destination_path)
                print(f"Successfully processed: {source_filename} -> {new_filename}")
                processed_files.append((source_filename, new_filename))
                
            except Exception as e:
                print(f"Error processing {source_filename}: {e}")
        else:
            print(f"File not found: {source_filename}")

        # process cNMF files for i
        for i in range(len):
            source_filename = f"{file_name_output}_{i}.df.npz"
    
    return processed_files

def rename_all_NMF(file_name_input, file_name_output, source_folder, destination_folder, len = 10, components = [30, 50, 60, 80, 100, 200, 250, 300]):

    for k in components:

        file_name_input_new = f"{file_name_input}.spectra.k_{k}.iter"
        file_name_output_new = f"{file_name_output}.spectra.k_{k}.iter"

        # With Inference/ subfolder, cnmf_tmp is at {source_folder}_{k}/Inference/cnmf_tmp
        source_folder_new = f"{source_folder}_{k}/Inference/cnmf_tmp"

        rename_and_move_files_NMF(file_name_input_new, file_name_output_new, source_folder_new, destination_folder)


In [ ]:
# With Inference/ subfolder: each parallel K run stores results in {run_name}_{K}/Inference/cnmf_tmp/
# and files are prefixed "Inference." (the cNMF name)
source_dir_base = "/oak/stanford/groups/engreitz/Users/ymo/Ronghao_100K_sample/Results/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par"
dest_dir = "/oak/stanford/groups/engreitz/Users/ymo/Ronghao_100K_sample/Results/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par/Inference_all/cnmf_tmp"

file_name_input = "Inference"
file_name_output = "Inference_all"

rename_all_NMF(file_name_input, file_name_output, source_dir_base, dest_dir, components = [30, 50, 60, 80, 100, 200, 250, 300])

# Comile consensus files

In [ ]:
# Transfer over consensus step
def consolidate_parallel_files(base_name, source_base_dir, dest_dir, 
                             k_values=[30, 50, 60, 80, 100, 200, 250, 300]):
    """
    Consolidates files from parallel cNMF runs into a single directory.
    With the Inference/ subfolder structure, each parallel K run stores
    consensus files in {source_base_dir}_{K}/Inference/.
    
    Parameters:
    - base_name: cNMF name used during inference (now 'Inference')
    - source_base_dir: Base directory containing the _K subdirectories  
    - dest_dir: Destination directory for consolidated files
    - k_values: List of K values to process
    """
    
    # Create destination folder if it doesn't exist
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    
    processed_files = []
    skipped_files = []
    
    # File patterns to include (exclude k_selection files)
    include_patterns = [
        'clustering',
        'gene_spectra_score', 
        'gene_spectra_tpm',
        'spectra',  # includes spectra consensus files
        'starcat_spectra',
        'usages.k_'    # includes usages consensus files
    ]
    
    # File patterns to exclude
    exclude_patterns = [
        'overdispersed_genes',
        'k_selection',
        'config_',
        '.yml'
    ]
    
    for k in k_values:
        # With Inference/ subfolder, consensus files are in {base}_{k}/Inference/
        source_dir = f"{source_base_dir}_{k}/Inference"
        
        if not os.path.exists(source_dir):
            print(f"Warning: Directory {source_dir} not found, skipping K={k}")
            continue
            
        print(f"Processing K={k} from {source_dir}")
        
        # Get all files in the source directory
        try:
            files = os.listdir(source_dir)
        except OSError as e:
            print(f"Error reading directory {source_dir}: {e}")
            continue
            
        for filename in files:
            # Skip directories
            if os.path.isdir(os.path.join(source_dir, filename)):
                continue
                
            # Skip if file matches exclude patterns
            if any(pattern in filename for pattern in exclude_patterns):
                skipped_files.append(f"{source_dir}/{filename}")
                continue
                
            # Only process files that match include patterns
            if any(pattern in filename for pattern in include_patterns):
                source_path = os.path.join(source_dir, filename)
                
                # Files are named Inference.clustering.k_30... etc.
                # Keep as-is or rename to consolidated name
                new_filename = filename
                    
                dest_path = os.path.join(dest_dir, new_filename)
                
                try:
                    shutil.copy2(source_path, dest_path)
                    print(f"  Copied: {filename} -> {new_filename}")
                    processed_files.append((source_path, dest_path))
                    
                except Exception as e:
                    print(f"  Error copying {filename}: {e}")
            else:
                skipped_files.append(f"{source_dir}/{filename}")
    
    print(f"\nSummary:")
    print(f"  Processed files: {len(processed_files)}")
    print(f"  Skipped files: {len(skipped_files)}")
    print(f"  Files saved to: {dest_dir}")
    
    if skipped_files:
        print(f"\nSkipped files (k_selection, config, etc.):")
        for f in skipped_files[:10]:  # Show first 10 skipped files
            print(f"  {f}")
        if len(skipped_files) > 10:
            print(f"  ... and {len(skipped_files) - 10} more")
    
    return processed_files, skipped_files

In [ ]:
# Example usage with your data
base_name = "Inference"
source_base_dir = "/oak/stanford/groups/engreitz/Users/ymo/Ronghao_100K_sample/Results/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par"
dest_dir = "/oak/stanford/groups/engreitz/Users/ymo/Ronghao_100K_sample/Results/120825_100k_cells_10iter_allhvg_torch_halsvar_batch_e7_h100_par/consolidated_results"

# Run the consolidation
processed, skipped = consolidate_parallel_files(base_name, source_base_dir, dest_dir)

# combine 2 different cNMF into one

In [3]:
def rename_and_move_files(k, file_name_input, file_name_output, source_dir, dest_dir, len = 10, second = False):
    
    # Create destination folder if it doesn't exist
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    
    processed_files = []
    
    # Process files for i 
    for i in range(len):

        source_file = f"{file_name_input}.spectra.k_{k}.iter_{i}.df.npz"
        source_path = os.path.join(source_dir, source_file )

        # Check if source file exists
        if os.path.exists(source_dir): 

            if second: 
                i = i + 10
            
            # Create new filename
            new_filename = f"{file_name_output}.spectra.k_{k}.iter_{i}.df.npz"
            destination_path = os.path.join(dest_dir, new_filename)
            
            try:
                # Copy and rename file to destination
                shutil.copy2(source_path, destination_path)
                print(f"Successfully processed: {source_file} -> {new_filename}")
                processed_files.append((source_file , new_filename))
                
            except Exception as e:
                print(f"Error processing {file_name_input}: {e}")
        else:
            print(f"File not found: {file_name_input}")
            print(source_path)
    
    return processed_files


def rename_all(file_name_input, file_name_output, source_dir, dest_dir, components = [30, 50, 60, 80, 100, 200, 250, 300], second = False):

    for k in components:

        data = rename_and_move_files(k, file_name_input, file_name_output, source_dir, dest_dir, second = second)


In [ ]:
# With Inference/ subfolder, cnmf_tmp is under Inference/
source_dir = "/oak/stanford/groups/engreitz/Users/ymo/NMF_re-inplementing/Results/torch-cNMF_evaluation/091625_100k_cells_10iter_torch_halsvar_online_e7/Inference/cnmf_tmp"
dest_dir = "/oak/stanford/groups/engreitz/Users/ymo/NMF_re-inplementing/Results/Consensus_across_methods/online_halsvar_cd_e7/Inference/cnmf_tmp"

file_name_input = "Inference"
file_name_output = "Inference"

rename_all(file_name_input, file_name_output, source_dir, dest_dir, components = [30, 50, 60, 80, 100, 200, 250, 300], second = True)